In [17]:

import re
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader
from docx import Document

# ==========================================================
# DOCUMENT QUESTION-ANSWERING ASSISTANT
# Single-cell Jupyter Notebook implementation
# ==========================================================

SUPPORTED_TYPES = {".txt", ".pdf", ".docx"}
MIN_RELEVANCE = 0.12
CHUNK_WORDS = 120
OVERLAP_WORDS = 25

document_text = ""
document_name = ""
chunks = []
vectorizer = None
document_matrix = None


# -------------------- Document reading --------------------

def read_document(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    if extension == ".txt":
        return path.read_text(encoding="utf-8", errors="replace")

    elif extension == ".pdf":
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    elif extension == ".docx":
        doc = Document(str(path))
        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

        for table in doc.tables:
            for row in table.rows:
                paragraphs.append(" | ".join(cell.text for cell in row.cells))

        return "\n".join(paragraphs)

    else:
        raise ValueError("Please upload a TXT, PDF, or DOCX file.")


# -------------------- Text chunking ----------------------

def split_into_chunks(text):
    words = re.findall(r"\S+", text)
    result = []

    if not words:
        return result

    step = CHUNK_WORDS - OVERLAP_WORDS

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + CHUNK_WORDS])

        if chunk.strip():
            result.append(chunk)

        if start + CHUNK_WORDS >= len(words):
            break

    return result


# -------------------- Document processing ----------------

def process_document(file_path):
    global document_text, document_name
    global chunks, vectorizer, document_matrix

    text = read_document(file_path).strip()

    if not text:
        raise ValueError(
            "No readable text was found. "
            "For scanned PDFs, OCR may be needed first."
        )

    document_text = text
    document_name = Path(file_path).name
    chunks = split_into_chunks(text)

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    document_matrix = vectorizer.fit_transform(chunks)

    return len(chunks)


# -------------------- Question answering -----------------



def answer_question(question):
    if not document_text or not document_text.strip():
        return {
            "answer": "Please upload and process a document first.",
            "sources": []
        }

    question = question.strip()

    if not question:
        return {"answer": "Please enter a question.", "sources": []}

    # Normalize common terms and abbreviations.
    normalized_question = question.lower()
    replacements = {
        r"\bai\b": "artificial intelligence",
        r"\bml\b": "machine learning",
        r"\bnlp\b": "natural language processing"
    }

    for pattern, replacement in replacements.items():
        normalized_question = re.sub(
            pattern, replacement, normalized_question
        )

    # Split the document into individual sentences.
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        document_text.replace("\n", " ")
    )
    sentences = [s.strip() for s in sentences if len(s.split()) >= 4]

    if not sentences:
        return {
            "answer": "Information not found in provided document.",
            "sources": []
        }

    # Handle summary requests using the first few meaningful sentences.
    if any(term in normalized_question for term in [
        "summarize", "summarise", "summary",
        "overview", "main points"
    ]):
        summary_sentences = sentences[:min(4, len(sentences))]

        return {
            "answer": " ".join(summary_sentences),
            "sources": [
                {
                    "passage_number": i + 1,
                    "relevance_score": 1.0,
                    "passage": sentence
                }
                for i, sentence in enumerate(summary_sentences)
            ]
        }

    # Rank individual sentences instead of large document chunks.
    try:
        sentence_vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )

        sentence_matrix = sentence_vectorizer.fit_transform(sentences)
        question_vector = sentence_vectorizer.transform(
            [normalized_question]
        )

        scores = cosine_similarity(
            question_vector, sentence_matrix
        ).flatten()

    except ValueError:
        return {
            "answer": "Information not found in provided document.",
            "sources": []
        }

    ranked_indices = scores.argsort()[::-1]

    # Keep only the best matching sentences.
    selected = [
        (int(i), float(scores[i]))
        for i in ranked_indices
        if scores[i] >= 0.08
    ][:2]

    if not selected:
        return {
            "answer": "Information not found in provided document.",
            "sources": []
        }

    # Concise answer: one or two relevant sentences only.
    answer = " ".join(sentences[i] for i, _ in selected)

    # Show the original matching sentences separately as evidence.
    sources = [
        {
            "passage_number": rank + 1,
            "relevance_score": round(score, 3),
            "passage": sentences[index]
        }
        for rank, (index, score) in enumerate(selected)
    ]

    return {
        "answer": answer,
        "sources": sources
    }


# -------------------- Interface --------------------------

title = widgets.HTML(
    value="""
    <div style="padding:18px;background:#202938;color:white;
                border-radius:10px;">
      <h2 style="margin:0;">Document Q&A Assistant</h2>
      <p style="margin:8px 0 0;">
        Upload a document, then ask questions based on its content.
      </p>
    </div>
    """
)

upload = widgets.FileUpload(
    accept=".txt,.pdf,.docx",
    multiple=False,
    description="Choose document"
)

load_button = widgets.Button(
    description="Process Document",
    button_style="primary",
    icon="upload"
)

status = widgets.Output()

question_box = widgets.Textarea(
    placeholder="Upload a document first, then type your question...",
    description="Question:",
    layout=widgets.Layout(width="100%", height="85px")
)

ask_button = widgets.Button(
    description="Ask Question",
    button_style="success",
    icon="search"
)

answer_output = widgets.Output()


def load_clicked(_):
    with status:
        clear_output()

        try:
            if not upload.value:
                print("Please choose a document first.")
                return

            # Supports current ipywidgets 8 and older upload formats.
            uploaded = upload.value

            if isinstance(uploaded, dict):
                item = next(iter(uploaded.values()))
                filename = item.get("name", "uploaded_document.txt")
                content = item["content"]
            else:
                item = uploaded[0]
                filename = item["name"]
                content = item["content"]

            suffix = Path(filename).suffix.lower()

            if suffix not in SUPPORTED_TYPES:
                print("Unsupported file type. Use TXT, PDF, or DOCX.")
                return

            # Use the actual extension so document parsers work correctly.
            temp_path = Path.cwd() / ("_uploaded_document" + suffix)
            temp_path.write_bytes(bytes(content))

            count = process_document(temp_path)

            print("Document processed successfully!")
            print("File:", document_name)
            print("Words:", len(document_text.split()))
            print("Passages created:", count)
            print("You can now ask questions below.")

        except Exception as error:
            print("Could not process the document:", error)


def ask_clicked(_):
    with answer_output:
        clear_output()

        if not document_text:
            display(Markdown(
                "**Please upload and process a document first.**"
            ))
            return

        result = answer_question(question_box.value)

        display(Markdown("### Question"))
        display(Markdown(question_box.value or "(No question entered)"))

        display(Markdown("### Answer"))
        display(Markdown(result["answer"]))

        if result["sources"]:
            display(Markdown("### Supporting source passages"))

            for source in result["sources"]:
                display(Markdown(
                    f"**Passage {source['passage_number']}** "
                    f"(TF-IDF similarity: "
                    f"{source['relevance_score']:.3f})"
                ))
            display(widgets.HTML(
                value=(
                    '<div style="background:#f1f5f9;color:#172033;'
                    'padding:12px;border-radius:8px;'
                    'border-left:4px solid #64748b;'
                    'white-space:pre-wrap;line-height:1.6;">'
                    + source["passage"].replace("&", "&amp;")
                    .replace("<", "&lt;").replace(">", "&gt;")
                    + '</div>'
                )
            ))

        else:
            display(Markdown(
                "**No sufficiently relevant source passage was found.**"
            ))


load_button.on_click(load_clicked)
ask_button.on_click(ask_clicked)

display(title)
display(widgets.HTML("<h3>1. Upload your document</h3>"))
display(upload)
display(load_button)
display(status)
display(widgets.HTML("<h3>2. Ask a question</h3>"))
display(question_box)
display(ask_button)
display(answer_output)


HTML(value='\n    <div style="padding:18px;background:#202938;color:white;\n                border-radius:10px…

HTML(value='<h3>1. Upload your document</h3>')

FileUpload(value=(), accept='.txt,.pdf,.docx', description='Choose document')

Button(button_style='primary', description='Process Document', icon='upload', style=ButtonStyle())

Output()

HTML(value='<h3>2. Ask a question</h3>')

Textarea(value='', description='Question:', layout=Layout(height='85px', width='100%'), placeholder='Upload a …

Button(button_style='success', description='Ask Question', icon='search', style=ButtonStyle())

Output()